# Intermediate Facilities Extraction for Surabaya

Notebook ini mengekstraksi titik fasilitas perantara (IF) sebagai kandidat lokasi pembuangan air oleh armada penyedot banjir. IF didefinisikan sebagai titik perpotongan antara badan air (sungai, kanal, saluran drainase) dengan jalan berkategori arteri atau kolektor (trunk, primary, secondary) di wilayah Kota Surabaya.

Output akhir adalah file CSV berisi koordinat IF beserta nama sungai dan jalan yang berpotongan pada tiap titik, siap dipakai sebagai input model MDCVRP-IF-SI.

## Overview

**Objective.** Menghasilkan dataset titik fasilitas perantara di Kota Surabaya dari OpenStreetMap.

**Pipeline.**
1. Query OSM Overpass API untuk semua waterway dan highway (trunk/primary/secondary) di bounding box Surabaya.
2. Parse hasil ke GeoDataFrame.
3. Filter: buang segmen jalan bertag `bridge=yes` karena jalan di atas jembatan tidak dapat dipakai membuang air ke sungai. Buang segmen sungai bertag `tunnel=yes` karena sungai bawah tanah tidak dapat diakses.
4. Reproject ke sistem koordinat metrik (UTM 49S / EPSG:32749) untuk operasi jarak dalam meter.
5. Hitung perpotongan geometri sungai dan jalan.
6. Dedup titik yang berdekatan (radius <50m) menggunakan DBSCAN, karena dua titik yang sangat berdekatan secara operasional dianggap satu IF.
7. Visualisasi hasil pada peta interaktif.
8. Export CSV dengan skema: `id`, `latitude`, `longitude`, `waterway_name`, `highway_name`, `highway_type`.

**Bounding box Surabaya.** Kurang lebih (south, west, north, east) = (-7.365, 112.60, -7.14, 112.86). Bisa disesuaikan di section Setup.

## Setup

### Install Dependencies

Colab sudah punya sebagian besar library. Yang perlu tambahan hanya `geopandas` dan `folium` untuk operasi spasial dan visualisasi.

In [1]:
%pip install -q geopandas folium shapely

### Imports

Import dikelompokkan: standard library, third-party, dan tidak ada modul lokal untuk notebook ini.

In [2]:
# Standard library
import json
import time
from pathlib import Path

# Third-party
import numpy as np
import pandas as pd
import geopandas as gpd
import requests
import folium
from shapely.geometry import LineString, Point, MultiPoint
from sklearn.cluster import DBSCAN

### Configuration

Semua parameter yang mungkin diubah dikumpulkan di sini agar tidak tersebar di banyak cell.

In [7]:
# Bounding box Kota Surabaya (south, west, north, east)
BBOX = (-7.365, 112.60, -7.14, 112.86)

# Kategori waterway dan highway yang dipertimbangkan
WATERWAY_TYPES = ['river', 'canal', 'stream', 'drain']
HIGHWAY_TYPES = ['trunk', 'primary', 'secondary']

# Endpoint Overpass API dengan fallback mirror.
# Kalau main endpoint kena rate-limit atau down, coba mirror berikutnya.
OVERPASS_ENDPOINTS = [
    'https://overpass-api.de/api/interpreter',
    'https://overpass.kumi.systems/api/interpreter',
    'https://overpass.private.coffee/api/interpreter',
]

# User-Agent wajib diisi meaningful. Default requests library sering ditolak
# dengan HTTP 406 oleh Overpass karena masuk blacklist.
USER_AGENT = 'MDCVRP-IF-SI-TA/1.0 (student research; Surabaya flood pumping)'

# Coordinate reference systems
CRS_WGS84 = 'EPSG:4326'      # untuk input/output lat-lon
CRS_UTM49S = 'EPSG:32749'    # UTM zona 49S untuk operasi metrik di Surabaya

# Radius deduplication dalam meter
DEDUP_RADIUS_METERS = 50

# Random seed untuk reproducibility
SEED = 42
np.random.seed(SEED)

# Output path
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / 'intermediate_facilities_surabaya.csv'
OUTPUT_GEOJSON = OUTPUT_DIR / 'intermediate_facilities_surabaya.geojson'

## Data Loading

Query Overpass API dua kali: satu untuk waterway, satu untuk highway. Kedua query pakai bounding box yang sama.

### Overpass Query Helper

In [8]:
def build_overpass_query(feature_key, feature_values, bbox):
    """Build Overpass QL query untuk mengambil semua way dengan tag tertentu di dalam bbox.

    Parameters
    ----------
    feature_key : str
        Kunci tag OSM, misalnya 'waterway' atau 'highway'.
    feature_values : list of str
        Daftar nilai tag yang diterima.
    bbox : tuple
        (south, west, north, east)
    """
    south, west, north, east = bbox
    values_regex = '|'.join(feature_values)
    query = f'''
    [out:json][timeout:120];
    (
      way["{feature_key}"~"^({values_regex})$"]({south},{west},{north},{east});
    );
    out geom tags;
    '''
    return query.strip()


def fetch_overpass(query, max_retries=3, retry_wait=30):
    """Kirim query ke Overpass API dengan retry dan fallback mirror.

    Overpass sering menolak request tanpa User-Agent yang jelas (HTTP 406),
    kena rate limit (HTTP 429), atau timeout (HTTP 504). Fungsi ini
    mencoba tiap endpoint di OVERPASS_ENDPOINTS dengan retry berjenjang.
    """
    headers = {
        'User-Agent': USER_AGENT,
        'Accept': 'application/json',
    }
    last_response = None

    for endpoint in OVERPASS_ENDPOINTS:
        print(f'Trying endpoint: {endpoint}')
        for attempt in range(1, max_retries + 1):
            try:
                response = requests.post(
                    endpoint,
                    data={'data': query},
                    headers=headers,
                    timeout=180,
                )
                last_response = response
                if response.status_code == 200:
                    return response.json()
                print(f'  Attempt {attempt}: HTTP {response.status_code}')
            except requests.RequestException as e:
                # Network error (timeout, connection refused, dst) juga di-retry
                print(f'  Attempt {attempt}: {type(e).__name__}: {e}')
            time.sleep(retry_wait)
        print(f'  All retries exhausted, switching endpoint')

    if last_response is not None:
        last_response.raise_for_status()
    raise RuntimeError('All Overpass endpoints failed')

### Fetch Waterway Data

In [9]:
waterway_query = build_overpass_query('waterway', WATERWAY_TYPES, BBOX)
print(waterway_query)

[out:json][timeout:120];
    (
      way["waterway"~"^(river|canal|stream|drain)$"](-7.365,112.6,-7.14,112.86);
    );
    out geom tags;


In [10]:
waterway_raw = fetch_overpass(waterway_query)
print(f'Jumlah waterway elements: {len(waterway_raw["elements"])}')

Trying endpoint: https://overpass-api.de/api/interpreter
Jumlah waterway elements: 894


### Fetch Highway Data

In [11]:
highway_query = build_overpass_query('highway', HIGHWAY_TYPES, BBOX)
print(highway_query)

[out:json][timeout:120];
    (
      way["highway"~"^(trunk|primary|secondary)$"](-7.365,112.6,-7.14,112.86);
    );
    out geom tags;


In [12]:
highway_raw = fetch_overpass(highway_query)
print(f'Jumlah highway elements: {len(highway_raw["elements"])}')

Trying endpoint: https://overpass-api.de/api/interpreter
Jumlah highway elements: 2821


## Data Cleaning

Parse hasil Overpass ke GeoDataFrame, kemudian filter segmen yang tidak relevan.

### Parse Overpass Response to GeoDataFrame

In [13]:
def elements_to_gdf(elements):
    """Convert list of Overpass 'way' elements ke GeoDataFrame LineString.

    Elements yang tidak punya geometry atau kurang dari 2 titik dilewati.
    """
    records = []
    for el in elements:
        if el.get('type') != 'way' or 'geometry' not in el:
            continue
        coords = [(pt['lon'], pt['lat']) for pt in el['geometry']]
        if len(coords) < 2:
            continue
        tags = el.get('tags', {})
        records.append({
            'osm_id': el['id'],
            'name': tags.get('name'),
            'category': tags.get('waterway') or tags.get('highway'),
            'bridge': tags.get('bridge'),
            'tunnel': tags.get('tunnel'),
            'layer': tags.get('layer'),
            'geometry': LineString(coords),
        })
    gdf = gpd.GeoDataFrame(records, crs=CRS_WGS84)
    return gdf

In [14]:
waterway_gdf = elements_to_gdf(waterway_raw['elements'])
highway_gdf = elements_to_gdf(highway_raw['elements'])

print(f'Waterway segments : {len(waterway_gdf)}')
print(f'Highway segments  : {len(highway_gdf)}')

# Cek distribusi kategori
print('\nWaterway categories:')
print(waterway_gdf['category'].value_counts())
print('\nHighway categories:')
print(highway_gdf['category'].value_counts())

Waterway segments : 894
Highway segments  : 2821

Waterway categories:
category
river     561
stream    167
drain     142
canal      24
Name: count, dtype: int64

Highway categories:
category
secondary    1140
primary       991
trunk         690
Name: count, dtype: int64


### Filter Non-Usable Segments

Alasan filter:
- **Highway `bridge=yes`**: jalan yang berbentuk jembatan melintasi sungai tidak dapat dipakai membuang air (kendaraan tidak dapat menurunkan selang dari atas jembatan langsung ke sungai). Perpotongan geometri terjadi, tapi tidak relevan secara operasional.
- **Waterway `tunnel=yes`**: sungai/saluran bawah tanah tidak dapat diakses dari permukaan.
- **Layer negatif atau positif ekstrem**: menandakan crossing bertingkat (bukan pertemuan permukaan aktual).

In [15]:
def is_bridge(row):
    return row['bridge'] == 'yes'

def is_tunnel(row):
    return row['tunnel'] == 'yes'

def has_nonzero_layer(row):
    # Layer OSM biasanya string '0', '1', '-1', dst.
    layer = row['layer']
    if layer is None:
        return False
    try:
        return int(layer) != 0
    except ValueError:
        return False


# Filter highway: buang bridge dan layer bertingkat
highway_before = len(highway_gdf)
highway_clean = highway_gdf[
    ~highway_gdf.apply(is_bridge, axis=1)
    & ~highway_gdf.apply(has_nonzero_layer, axis=1)
].copy()
print(f'Highway: {highway_before} -> {len(highway_clean)} setelah filter')

# Filter waterway: buang tunnel dan layer bertingkat
waterway_before = len(waterway_gdf)
waterway_clean = waterway_gdf[
    ~waterway_gdf.apply(is_tunnel, axis=1)
    & ~waterway_gdf.apply(has_nonzero_layer, axis=1)
].copy()
print(f'Waterway: {waterway_before} -> {len(waterway_clean)} setelah filter')

# Validasi: pastikan masih ada data setelah filter
assert len(highway_clean) > 0, 'Tidak ada highway tersisa setelah filter'
assert len(waterway_clean) > 0, 'Tidak ada waterway tersisa setelah filter'

Highway: 2821 -> 2508 setelah filter
Waterway: 894 -> 687 setelah filter


### Reproject to Metric CRS

Operasi jarak (deduplication) memerlukan sistem koordinat metrik. UTM zona 49S (EPSG:32749) adalah zona yang mencakup Jawa Timur.

In [16]:
waterway_utm = waterway_clean.to_crs(CRS_UTM49S)
highway_utm = highway_clean.to_crs(CRS_UTM49S)

print(f'Waterway CRS: {waterway_utm.crs}')
print(f'Highway CRS : {highway_utm.crs}')

Waterway CRS: EPSG:32749
Highway CRS : EPSG:32749


## Spatial Intersection

Cari semua titik perpotongan geometri antara segmen sungai dan segmen jalan. Untuk efisiensi, pakai spatial index (R-tree) pada highway agar tidak perlu melakukan pairwise check O(n*m).

In [17]:
def compute_intersections(waterway_gdf, highway_gdf):
    """Hitung semua titik perpotongan antara waterway dan highway.

    Menghasilkan list of dict dengan geometri Point (dalam CRS input).
    Hasil intersection bisa berupa Point (crossing tunggal), MultiPoint
    (crossing di beberapa titik), atau LineString (segmen tumpang tindih,
    langka pada waterway vs highway, jadi diabaikan).
    """
    hw_sindex = highway_gdf.sindex
    intersections = []

    for _, wr in waterway_gdf.iterrows():
        candidate_idx = list(hw_sindex.intersection(wr.geometry.bounds))
        if not candidate_idx:
            continue
        for idx in candidate_idx:
            hr = highway_gdf.iloc[idx]
            inter = wr.geometry.intersection(hr.geometry)
            if inter.is_empty:
                continue

            # Ambil hanya geometri Point (crossing sesungguhnya)
            if inter.geom_type == 'Point':
                points = [inter]
            elif inter.geom_type == 'MultiPoint':
                points = list(inter.geoms)
            else:
                # LineString atau tipe lain dilewati
                continue

            for pt in points:
                intersections.append({
                    'geometry': pt,
                    'waterway_id': wr['osm_id'],
                    'waterway_name': wr['name'],
                    'waterway_type': wr['category'],
                    'highway_id': hr['osm_id'],
                    'highway_name': hr['name'],
                    'highway_type': hr['category'],
                })
    return intersections

In [18]:
raw_intersections = compute_intersections(waterway_utm, highway_utm)

# Bangun GeoDataFrame dalam CRS UTM (masih dalam meter)
inter_gdf = gpd.GeoDataFrame(raw_intersections, crs=CRS_UTM49S)
print(f'Titik perpotongan mentah (belum dedup): {len(inter_gdf)}')

Titik perpotongan mentah (belum dedup): 1


## Deduplication

Titik perpotongan yang berjarak sangat dekat (misal ketika sungai berpotongan dengan jalan yang bercabang di persimpangan) secara operasional dianggap satu IF. Pakai DBSCAN dengan `eps = DEDUP_RADIUS_METERS` dan `min_samples=1` sehingga setiap titik pasti masuk cluster (tidak ada yang diberi label -1/noise).

In [19]:
if len(inter_gdf) == 0:
    raise RuntimeError('Tidak ada titik perpotongan. Cek bounding box atau data mentah.')

coords = np.array([[pt.x, pt.y] for pt in inter_gdf.geometry])

# DBSCAN dalam ruang metrik (meter), eps = radius toleransi
clustering = DBSCAN(eps=DEDUP_RADIUS_METERS, min_samples=1).fit(coords)
inter_gdf['cluster'] = clustering.labels_

n_clusters = inter_gdf['cluster'].nunique()
print(f'Cluster hasil DBSCAN: {n_clusters}')
print(f'Rasio reduksi: {len(inter_gdf)} -> {n_clusters}')

Cluster hasil DBSCAN: 1
Rasio reduksi: 1 -> 1


In [20]:
def aggregate_names(series):
    """Gabungkan nama unik non-null jadi string dipisah koma."""
    unique_names = {str(v) for v in series if pd.notna(v)}
    return ', '.join(sorted(unique_names)) if unique_names else None


# Untuk tiap cluster, ambil titik pertama sebagai representatif,
# lalu agregasi info nama dan tipe dari semua titik di cluster
cluster_agg = inter_gdf.groupby('cluster').agg(
    geometry=('geometry', 'first'),
    waterway_name=('waterway_name', aggregate_names),
    waterway_type=('waterway_type', aggregate_names),
    highway_name=('highway_name', aggregate_names),
    highway_type=('highway_type', aggregate_names),
    n_source_points=('geometry', 'count'),
).reset_index(drop=True)

if_utm = gpd.GeoDataFrame(cluster_agg, geometry='geometry', crs=CRS_UTM49S)
print(f'Total IF setelah dedup: {len(if_utm)}')
if_utm.head()

Total IF setelah dedup: 1


,geometry,waterway_name,waterway_type,highway_name,highway_type,n_source_points
0,POINT (677120.586 9187190.284),None,drain,Ruas Jalan Petiken - Sumput,secondary,1


## Visualization

Plot titik IF pada peta interaktif Folium untuk visual sanity check. Overlay dengan sungai dan jalan supaya terlihat bahwa titik memang berada di perpotongan.

In [21]:
# Kembali ke CRS WGS84 untuk plotting
if_wgs = if_utm.to_crs(CRS_WGS84)
waterway_wgs = waterway_clean
highway_wgs = highway_clean

# Center peta pada centroid IF
center_lat = if_wgs.geometry.y.mean()
center_lon = if_wgs.geometry.x.mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=12, tiles='OpenStreetMap')

# Overlay waterway sebagai polyline biru
for _, row in waterway_wgs.iterrows():
    coords = [(pt[1], pt[0]) for pt in row.geometry.coords]
    folium.PolyLine(coords, color='#3388ff', weight=1.5, opacity=0.6).add_to(m)

# Overlay highway sebagai polyline abu
for _, row in highway_wgs.iterrows():
    coords = [(pt[1], pt[0]) for pt in row.geometry.coords]
    folium.PolyLine(coords, color='#666666', weight=1.5, opacity=0.5).add_to(m)

# Titik IF sebagai marker merah
for i, row in if_wgs.iterrows():
    popup_html = f"""
    <b>IF_{i:03d}</b><br>
    Waterway: {row['waterway_name'] or '(unnamed)'}<br>
    Highway : {row['highway_name'] or '(unnamed)'} ({row['highway_type']})<br>
    Source points: {row['n_source_points']}
    """
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=6,
        color='red',
        fill=True,
        fill_opacity=0.8,
        popup=folium.Popup(popup_html, max_width=300),
    ).add_to(m)

m

Output hidden; open in https://colab.research.google.com to view.

## Export

Simpan dua format:
- **CSV** untuk konsumsi oleh backend algoritma (mudah dibaca pandas).
- **GeoJSON** untuk konsumsi oleh frontend Leaflet (mudah di-load sebagai layer peta).

In [22]:
# Susun DataFrame untuk export CSV dengan skema yang stabil
export_df = pd.DataFrame({
    'id': [f'IF_{i:03d}' for i in range(len(if_wgs))],
    'latitude': if_wgs.geometry.y.values,
    'longitude': if_wgs.geometry.x.values,
    'waterway_name': if_wgs['waterway_name'].values,
    'waterway_type': if_wgs['waterway_type'].values,
    'highway_name': if_wgs['highway_name'].values,
    'highway_type': if_wgs['highway_type'].values,
    'n_source_points': if_wgs['n_source_points'].values,
})
export_df.head()

,id,latitude,longitude,waterway_name,waterway_type,highway_name,highway_type,n_source_points
0,IF_000,-7.350469,112.604635,None,drain,Ruas Jalan Petiken - Sumput,secondary,1


In [23]:
export_df.to_csv(OUTPUT_CSV, index=False)
print(f'Saved: {OUTPUT_CSV}')

# GeoJSON dengan properties yang sama
if_export_gdf = gpd.GeoDataFrame(
    export_df.drop(columns=['latitude', 'longitude']),
    geometry=gpd.points_from_xy(export_df['longitude'], export_df['latitude']),
    crs=CRS_WGS84,
)
if_export_gdf.to_file(OUTPUT_GEOJSON, driver='GeoJSON')
print(f'Saved: {OUTPUT_GEOJSON}')

Saved: output/intermediate_facilities_surabaya.csv
Saved: output/intermediate_facilities_surabaya.geojson


### Download di Google Colab

Cell berikut hanya berjalan di lingkungan Colab. Di lokal, file sudah ada di folder `output/`.

In [24]:
try:
    from google.colab import files
    files.download(str(OUTPUT_CSV))
    files.download(str(OUTPUT_GEOJSON))
except ImportError:
    print('Bukan di Google Colab, download manual dari folder output/')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Conclusion

Notebook menghasilkan dataset IF Kota Surabaya berupa titik perpotongan sungai dengan jalan arteri/kolektor, sudah difilter dari jembatan dan tunnel, dan didedup pada radius 50m.

**Output files.**
- `output/intermediate_facilities_surabaya.csv` untuk backend algoritma.
- `output/intermediate_facilities_surabaya.geojson` untuk frontend Leaflet.

**Skema kolom.**
- `id`: identifier IF (IF_000, IF_001, dst)
- `latitude`, `longitude`: koordinat WGS84
- `waterway_name`, `waterway_type`: nama dan tipe sungai
- `highway_name`, `highway_type`: nama dan tipe jalan
- `n_source_points`: jumlah titik intersection mentah yang dikelompokkan ke IF ini (indikator kepadatan crossing)

**Next steps.**
1. Verifikasi visual pada peta: apakah titik IF sesuai ekspektasi (tidak jatuh di tengah sungai, tidak jatuh di area yang jelas tidak accessible)?
2. Kalau ada IF yang tidak masuk akal, revisi filter atau tambahkan blacklist manual.
3. Gabungkan dengan matriks jarak OSRM: hitung jarak dari tiap titik genangan dan depo ke tiap IF.
4. Integrasikan ke pipeline model MDCVRP-IF-SI sebagai himpunan R (fasilitas perantara).